# K2 — Transmission and mixing

This page checks phase K2 of the port: the generalised force of infection, infectiousness
weights, and the time-varying age mixing matrix.

**Claims to check.**

1. The port's yearly mixing matrices equal the original builder's (which looped over 36
   age-band pairs inside the vector field). The heatmaps should look the same and the
   assertion checks them to 1e-12.
2. With homogeneous mixing (`homog_baseline`) and with age mixing (`hetero_baseline`), all 160
   compartments match the original at every year to 1e-5 relative (floor 1e-6 persons).

The two runs differ visibly: age mixing concentrates infection in adults, so the two
epidemics are not the same curve. That is what makes the second claim a test of the mixing.

In [ ]:
import sys

import numpy as np
import pandas as pd
import plotly.express as px
import plotly.io as pio
import yaml

pd.options.plotting.backend = "plotly"
pio.renderers.default = "notebook_connected"

from kiribati_tb.demography import AGE_GROUPS
from kiribati_tb.mixing import START_YEAR, read_conmat_matrix, yearly_matrices
from kiribati_tb.model import ACTIVE_COMPS, ModelConfig, compile_model, run_compartments
from kiribati_tb.paths import GOLDEN, REPO_ROOT

sys.path.insert(0, str(REPO_ROOT / "tests"))
from parity import parity_report, read_golden

params = yaml.safe_load((GOLDEN / "hetero_baseline" / "params.yaml").read_text())["params"]

## Mixing matrices

Rows are the infected age band, columns the infecting band. Each yearly matrix is normalised
to spectral radius one with `eigvalsh` on the symmetric form. Left: the port in 1900 and 2025;
right: the `conmat` matrix the calibration compares against.

In [ ]:
stack = np.asarray(yearly_matrices(params))
panels = {
    "port, 1900": stack[1900 - START_YEAR],
    "port, 2025": stack[2025 - START_YEAR],
    "conmat (target)": read_conmat_matrix(),
}
fig = px.imshow(
    np.stack(list(panels.values())),
    facet_col=0,
    x=list(AGE_GROUPS),
    y=list(AGE_GROUPS),
    labels={"x": "infecting age band", "y": "infected age band", "color": "contact"},
    title="Normalised age mixing matrices",
)
for i, name in enumerate(panels):
    fig.layout.annotations[i].text = name
fig

The original builder's matrices at ten times for five parameter draws are stored in
`tests/golden/mixing.parquet`. The assertion compares every one.

In [ ]:
golden_mixing = pd.read_parquet(GOLDEN / "mixing.parquet")
cols = [f"m_{i}_{j}" for i in range(8) for j in range(8)]
worst = 0.0
for (bg, spread, pc), rows in golden_mixing.groupby(["bg_mixing", "a_spread", "pc_strength"]):
    s = np.asarray(yearly_matrices({"bg_mixing": bg, "a_spread": spread, "pc_strength": pc}))
    got = s[np.floor(rows["time"].to_numpy()).astype(int) - START_YEAR].reshape(len(rows), -1)
    worst = max(worst, float(np.max(np.abs(got - rows[cols].to_numpy()))))
print(f"max |port - original| over 50 matrices: {worst:.2e}")
assert worst < 1e-12

## Homogeneous vs age-structured transmission

Active TB (the four disease states, all ages) under each mixing assumption: port as solid
lines, original as dashed. Each pair should overlap; the two pairs should not.

In [ ]:
runs = {}
for fixture, config in {
    "homog_baseline": ModelConfig(heterogeneous_mixing=False),
    "hetero_baseline": ModelConfig(heterogeneous_mixing=True),
}.items():
    p = yaml.safe_load((GOLDEN / fixture / "params.yaml").read_text())["params"]
    gold = read_golden(fixture, "compartments")
    runs[fixture] = (run_compartments(compile_model(config), p)[gold.columns], gold)


def active(df: pd.DataFrame) -> pd.Series:
    return df[[c for c in df.columns if c.split("X")[0] in ACTIVE_COMPS]].sum(axis=1)


lines = pd.DataFrame(
    {
        f"{fixture}, {source}": active(frame)
        for fixture, (got, gold) in runs.items()
        for source, frame in (("summer4", got), ("summer2gen", gold))
    }
)
fig = lines.plot(
    title="People with active TB: summer4 port vs original",
    labels={"index": "year", "value": "people", "variable": "run"},
)
for trace in fig.data:
    if "summer2gen" in trace.name:
        trace.line.dash = "dash"
fig

## Error per compartment

The largest relative error per compartment for each fixture (floor 1e-6 persons). The
assertion is the parity claim.

In [ ]:
reports = {fixture: parity_report(got, gold, atol=1e-6) for fixture, (got, gold) in runs.items()}
for fixture, report in reports.items():
    assert report["passes"].all(), (fixture, report[~report["passes"]])
pd.DataFrame({fixture: r["max_rel"] for fixture, r in reports.items()}).plot.box(
    title="Max relative error per compartment (all 160)",
    labels={"variable": "fixture", "value": "max relative error"},
    log_y=True,
)